In [1]:
import numpy as np
import cv2
import skimage.feature as sk
#import matplotlib.pyplot as plt

In [2]:
# Load the 2 images
left_img = cv2.imread('left_im.png')
center_img = cv2.imread('center_im.png')
right_img = cv2.imread('right_im.png')

images = [left_img, center_img, right_img]

In [3]:
left_img.shape

(774, 414, 3)

In [4]:
# harrys corner detection
def corner_harris(image, block_size, ksize, k):
    # Convert image to gray scale
    gray_im = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_im = np.float32(gray_im)

    # Compute the corner response function for each pixel in the image
    features_im = cv2.cornerHarris(gray_im, block_size, ksize, k)

    # Dilate the corner response function to enhance the corner points
    features_im = cv2.dilate(features_im, None)

    # Need create a new instance of the image, otherwise we cant run the test with different parameters
    # Has it will modify the original image and the next test will be run on the modified image
    image_copy = image.copy()

    # Mark the corners in the original image with a red pixel
    image_copy[features_im > 0.01 * features_im.max()] = [255, 0, 0]
    harris_features = features_im > 0.01 * features_im.max()

    return image_copy, harris_features

# Investigating the effect of the parameters on the size and number of detected corners
"""
test1_1_left, _ = corner_harris(left_img, 17, 21, 0.01)
test1_2_left, _ = corner_harris(left_img, 9, 13, 0.01)
test1_3_left, _ = corner_harris(left_img, 5, 7, 0.01)
test1_4_left, _ = corner_harris(left_img, 3, 5, 0.01)

cv2.imshow('test1_1_left', test1_1_left)
cv2.imshow('test1_2_left', test1_2_left)
cv2.imshow('test1_3_left', test1_3_left)
cv2.imshow('test1_4_left', test1_4_left)
cv2.waitKey(0)
cv2.destroyAllWindows()

test2_1_left, _ = corner_harris(left_img, 5, 7, 0.01)
test2_2_left, _ = corner_harris(left_img, 5, 7, 0.04)
test2_3_left, _ = corner_harris(left_img, 5, 7, 0.06)
test2_4_left, _ = corner_harris(left_img, 5, 7, 0.1)

cv2.imshow('test2_1_left', test2_1_left)
cv2.imshow('test2_2_left', test2_2_left)
cv2.imshow('test2_3_left', test2_3_left)
cv2.imshow('test2_4_left', test2_4_left)
cv2.waitKey(0)
cv2.destroyAllWindows()
"""

"\ntest1_1_left, _ = corner_harris(left_img, 17, 21, 0.01)\ntest1_2_left, _ = corner_harris(left_img, 9, 13, 0.01)\ntest1_3_left, _ = corner_harris(left_img, 5, 7, 0.01)\ntest1_4_left, _ = corner_harris(left_img, 3, 5, 0.01)\n\ncv2.imshow('test1_1_left', test1_1_left)\ncv2.imshow('test1_2_left', test1_2_left)\ncv2.imshow('test1_3_left', test1_3_left)\ncv2.imshow('test1_4_left', test1_4_left)\ncv2.waitKey(0)\ncv2.destroyAllWindows()\n\ntest2_1_left, _ = corner_harris(left_img, 5, 7, 0.01)\ntest2_2_left, _ = corner_harris(left_img, 5, 7, 0.04)\ntest2_3_left, _ = corner_harris(left_img, 5, 7, 0.06)\ntest2_4_left, _ = corner_harris(left_img, 5, 7, 0.1)\n\ncv2.imshow('test2_1_left', test2_1_left)\ncv2.imshow('test2_2_left', test2_2_left)\ncv2.imshow('test2_3_left', test2_3_left)\ncv2.imshow('test2_4_left', test2_4_left)\ncv2.waitKey(0)\ncv2.destroyAllWindows()\n"

In [5]:
harris_output = [corner_harris(img, 5, 7, 0.06) for img in images] 
harris_RGB_image = [cv2.cvtColor(x[0], cv2.COLOR_BGR2RGB) for x in harris_output]


In [6]:
def compute_keyPoints(harris_features): 
    keypoints = []
    for i in range(harris_features.shape[0]):
        for j in range(harris_features.shape[1]):
            if harris_features[i, j]:
                keypoints.append(cv2.KeyPoint(j, i, size=16))
    return keypoints

In [15]:
# Compute SIFT features
# First exctract the key points in each image
key_points = [compute_keyPoints(x[1]) for x in harris_output]

# Then compute the SIFT features for each image using the key points
sift = cv2.xfeatures2d.SIFT_create()
SIFT_features = [sift.compute(img, keypoints) for img, keypoints in zip(images, key_points)]


In [19]:
key_points[0][0].pt

(398.0, 287.0)

In [ ]:
#Compute the distances between every descriptor in image 1 with every descriptor in image 2. For this, use: 
# a) Normalized correlation 
# b) Euclidean distance after normalizing each descriptor.

def normalize_descriptors(descriptors):
    norms = np.linalg.norm(descriptors, axis=1, keepdims=True)
    return descriptors / norms


def euclidean_distance(desc1, desc2):
    list = [:length(desc1), :length(desc2)]
    for e1 in desc1:
        for e2 in desc2:
            list.append(np.linalg.norm(e1 - e2))

    return np.linalg.norm(diff, axis=2)


def normalized_correlation(desc1, desc2):
    return np.dot(desc1, desc2.T)

normalized_SIFT_features = [normalize_descriptors(x[1]) for x in SIFT_features]

euclidean_distances = euclidean_distance(normalized_SIFT_features[0], normalized_SIFT_features[1])
normalized_correlations = normalized_correlation(normalized_SIFT_features[0], normalized_SIFT_features[1])

print("Euclidean Distances:\n", euclidean_distances)
print("Normalized Correlations:\n", normalized_correlations)

MemoryError: Unable to allocate 274. GiB for an array with shape (20412, 28127, 128) and data type float32